# 04 · Herramientas del agente y evaluador

**Qué contiene.** Celda 0: las herramientas del agente, el evaluador y una validación con un agente oráculo (responde siempre bien por construcción), sin GPU ni costo. Celda 1: prueba de conexión con Langfuse.

| | |
|---|---|
| **Runtime** | CPU (sin GPU) |
| **Entradas (`MyDrive/qlora-series-ar/data/`)** | `catalogo_busqueda.csv`, `eval_{split}_final2.csv`, `specs_{split}_v2.csv` |
| **Salidas** | Caché de la API (`cache_api.db`) |
| **Secretos de Colab** | `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` (solo la celda 1) |
| **Costo y tiempo** | Sin costo ni GPU. Minutos |
| **Orden** | Después de `02`; antes de `05` |
| **En el repositorio** | `herramientas.py`, `evaluador.py`, `agente.py` |

**Advertencias.**
- La celda 1 envía una traza de prueba a Langfuse y necesita las dos claves.


In [ ]:
print("CELDA: HERRAMIENTAS DEL AGENTE + EVALUADOR, VALIDADOS CON UN AGENTE ORÁCULO (CPU, SIN COSTO)")
import os, re, json, shutil, sqlite3, unicodedata
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
for nombre in ["cache_api.db", "busqueda.db"]:
    if not os.path.exists(f"/content/{nombre}"):
        shutil.copy(f"{RUTA}/{nombre}", f"/content/{nombre}")

cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}
PARAMS_TOOL = {"ids", "start_date", "end_date", "representation_mode", "limit", "sort"}

def clave(params):
    p = {k: str(v) for k, v in params.items() if v is not None}
    return json.dumps(dict(sorted(p.items())), ensure_ascii=False)

# ---------- Herramientas (solo caché; la consulta en vivo se agrega en el paso siguiente) ----------
class Herramientas:
    def __init__(self):
        self.cache = sqlite3.connect("/content/cache_api.db", check_same_thread=False)
        self.fts = sqlite3.connect("/content/busqueda.db", check_same_thread=False)
        self.fallos_cache = 0

    def search_series(self, query, k=10):
        toks = [t for t in re.findall(r"[a-z0-9]+", norm(query)) if t not in STOP and len(t) > 1]
        toks = list(dict.fromkeys(toks))
        if not toks:
            return "[]"
        partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
        rows = self.fts.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                                "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
        cand = [{"serie_id": s, "titulo": str(cat.loc[s, "serie_titulo"]),
                 "distribucion": str(cat.loc[s, "distribucion_titulo"]),
                 "frecuencia": str(cat.loc[s, "frecuencia"])} for s, _ in rows]
        return json.dumps(cand, ensure_ascii=False)

    def get_series(self, args):
        extra = set(args) - PARAMS_TOOL
        if extra or "ids" not in args:
            return json.dumps({"status": 400, "error": f"Parámetros no permitidos o faltantes: {sorted(extra) or ['ids']}"},
                              ensure_ascii=False)
        fila = self.cache.execute("SELECT status, cuerpo FROM cache WHERE clave=?", (clave(args),)).fetchone()
        if not fila:
            self.fallos_cache += 1
            return json.dumps({"status": None, "error": "consulta no disponible en la caché"}, ensure_ascii=False)
        status, cuerpo = fila
        js = json.loads(cuerpo)
        if status != 200:
            return json.dumps({"status": status, "error": js.get("errors", js)}, ensure_ascii=False)
        sid = str(args["ids"])
        c = cat.loc[sid] if sid in cat.index else None
        return json.dumps({"status": 200, "serie_id": sid,
                           "titulo": None if c is None else str(c["serie_titulo"]),
                           "unidades": None if (c is None or pd.isna(c["serie_unidades"])) else c["serie_unidades"],
                           "frecuencia": None if c is None else str(c["frecuencia"]),
                           "data": js.get("data", [])}, ensure_ascii=False)

# ---------- Evaluador ----------
def periodo(f, freq):
    f = "" if f is None else str(f)
    return f[:4] if freq == "anual" else f[:7]

def valor_fecha(tipo, freq, d, params):
    if tipo == "sin_datos" or not d or d[0][1] is None or d[-1][1] is None:
        return None, None
    if tipo == "ultimo_dato":
        return d[0][1], d[0][0]
    if tipo == "valor_puntual":
        return d[0][1], params["start_date"]
    return round(d[-1][1] * 100, 2), d[-1][0]          # variacion_interanual, en porcentaje

def cerca(a, b, tol_abs):
    return abs(a - b) <= max(tol_abs, 1e-3 * abs(b))

def puntuar(tipo, freq, g, sub, ids_ultima, objetivo):
    r = {"termino": sub is not None, "serie_ok": ids_ultima == objetivo}
    if sub is None:
        r.update(valor_ok=False, valor_flex=False, fecha_ok=False)
    elif tipo == "sin_datos":
        nulo = sub.get("valor") is None
        r.update(valor_ok=nulo, valor_flex=nulo, fecha_ok=sub.get("fecha") is None)
    else:
        v = sub.get("valor")
        num = isinstance(v, (int, float)) and not isinstance(v, bool)
        tol = 0.011 if tipo == "variacion_interanual" else 5e-5
        ok_e = bool(num and cerca(v, g["valor"], tol))
        ok_f = ok_e or bool(tipo == "variacion_interanual" and num and cerca(v, g["valor"] / 100, 1e-4))
        r.update(valor_ok=ok_e, valor_flex=ok_f,
                 fecha_ok=sub.get("fecha") is not None and periodo(sub["fecha"], freq) == periodo(g["fecha"], freq))
    r["correcta"] = bool(r["termino"] and r["serie_ok"] and r["valor_ok"] and r["fecha_ok"])
    r["correcta_flex"] = bool(r["termino"] and r["serie_ok"] and r["valor_flex"] and r["fecha_ok"])
    return r

# ---------- Oráculo y controles negativos ----------
H = Herramientas()
POLITICAS = ["oraculo", "valor_malo", "fecha_mala", "serie_mala"]

for split in ["val", "test"]:
    S = pd.read_csv(f"{RUTA}/specs_{split}_v2.csv")
    S["idx"] = S.index
    F = pd.read_csv(f"{RUTA}/eval_{split}_final2.csv")
    F["idx"] = F["spec_id"].str.split("-").str[1].astype(int)
    M = F.merge(S[["idx", "data"]], on="idx", how="left")
    filas, invalidos = [], 0
    for r in M.to_dict("records"):
        params, d_gold = json.loads(r["params"]), json.loads(r["data"])
        gv, gf = valor_fecha(r["tipo"], r["frecuencia"], d_gold, params)
        if r["tipo"] != "sin_datos" and gv is None:
            invalidos += 1
            continue
        g = {"valor": gv, "fecha": gf}
        res = json.loads(H.get_series(params))
        d = res.get("data", []) if res.get("status") == 200 else []
        tipo_ef = "valor_puntual" if r["tipo"] == "sin_datos" else r["tipo"]
        for pol in POLITICAS:
            v, f = valor_fecha(tipo_ef, r["frecuencia"], d, params)
            ids_ultima = params["ids"]
            if pol == "valor_malo":
                v = 1.0 if v is None else v * 1.5 + 1
            elif pol == "fecha_mala":
                f = "1900-01"
            elif pol == "serie_mala":
                ids_ultima = "otra_serie"
            sc = puntuar(r["tipo"], r["frecuencia"], g, {"valor": v, "fecha": f}, ids_ultima, params["ids"])
            filas.append({"id": r["spec_id"], "tipo": r["tipo"], "politica": pol, **sc})
    D = pd.DataFrame(filas)
    print(f"\n=== {split}: {len(M)} especificaciones | gold inválido (excluidas): {invalidos} | "
          f"evaluadas: {D['id'].nunique()} | fallos de caché: {H.fallos_cache}")
    t = D.pivot_table(index="politica", columns="tipo", values="correcta", aggfunc="mean")
    t["TODAS"] = D.groupby("politica")["correcta"].mean()
    print("% correcta por política y tipo:")
    print((t * 100).round(1).to_string())
    print("Oráculo, versión flexible:", f"{D[D['politica'] == 'oraculo']['correcta_flex'].mean():.1%}")
    malos = D[(D["politica"] == "oraculo") & ~D["correcta"]]
    if len(malos):
        print("Casos donde el oráculo falla:")
        print(malos.head(8)[["id", "tipo", "serie_ok", "valor_ok", "fecha_ok"]].to_string(index=False))

# ---------- Ejemplos del formato de las herramientas ----------
print("\nget_series (variación):", H.get_series(json.loads(M[M["tipo"] == "variacion_interanual"].iloc[0]["params"]))[:330])
print("search_series('tipo de cambio'):", H.search_series("tipo de cambio")[:330])

In [ ]:
!pip -q install -U langfuse
print("CELDA: PRUEBA DE CONEXIÓN CON LANGFUSE (v2)")
import os
from importlib.metadata import version
from google.colab import userdata

HOST = "https://us.cloud.langfuse.com"
pk = userdata.get("LANGFUSE_PUBLIC_KEY").strip()
sk = userdata.get("LANGFUSE_SECRET_KEY").strip()
os.environ["LANGFUSE_PUBLIC_KEY"] = pk
os.environ["LANGFUSE_SECRET_KEY"] = sk
os.environ["LANGFUSE_HOST"] = HOST          # nombre usado en versiones anteriores
os.environ["LANGFUSE_BASE_URL"] = HOST      # nombre usado en versiones recientes
print("Host configurado:", os.environ["LANGFUSE_HOST"])

from langfuse import Langfuse, get_client
print("SDK:", version("langfuse"))

lf = None
intentos = [
    ("get_client() con variables de entorno", lambda: get_client()),
    ("Langfuse(host=...)", lambda: Langfuse(public_key=pk, secret_key=sk, host=HOST)),
    ("Langfuse(base_url=...)", lambda: Langfuse(public_key=pk, secret_key=sk, base_url=HOST)),
]
for nombre, fabrica in intentos:
    try:
        cliente = fabrica()
    except Exception as e:
        print(f"  {nombre}: no se pudo crear ({type(e).__name__}: {str(e)[:80]})")
        continue
    try:
        ok = cliente.auth_check()
        print(f"  {nombre}: auth_check -> {ok}")
    except Exception as e:
        print(f"  {nombre}: {type(e).__name__}")
        continue
    if ok:
        lf = cliente
        print("  >> funciona con:", nombre)
        break
assert lf is not None, "Ninguna forma de conexión funcionó: pegame este output"

with lf.start_as_current_observation(as_type="span", name="prueba-agente",
                                     input={"pregunta": "hola"}) as span:
    tid = lf.get_current_trace_id()
    with lf.start_as_current_observation(as_type="generation", name="llm-simulado",
                                         model="prueba") as gen:
        gen.update(output="respuesta simulada")
    with lf.start_as_current_observation(as_type="span", name="herramienta-simulada",
                                         input={"consulta": "tipo de cambio"}) as herr:
        herr.update(output={"resultados": 3})
    span.update(output={"ok": True})

lf.flush()
print("Traza enviada. id:", tid)
try:
    print("URL:", lf.get_trace_url(trace_id=tid))
except Exception as e:
    print("No pude armar la URL (no es grave):", type(e).__name__)